# Generating a Random Stew of Newspaper Text

In this notebook, I'm building a "random stew" of newspaper text from Chronicling America. Why? Because I need negative examples to fine-tune BERT (i.e., examples of text that do not contain references to local racial violence). A sample of this sort of material is necessary because, otherwise, how else is BERT to know what I'm _not_ looking for? My hand-keyed data from 02_identify_local_lynch_coverage.ipynb and 03_tag_local_reports.ipynb will tell BERT what kinds of texts I'm looking for, but the model will need negative examples, too. Otherwise, it will have no baseline for comparison.

While the purpose of this random stew is straightforward enough, compiling an appropriately random sample of newspaper texts requires some thoughtful execution. For one thing, how do you make the sample truly random? And, how do you ensure the sample offers appropriate counter-examples to my positive training set (the hand-keyed reports of local coverage)?

To these ends, I've put together the following steps. They review my hand-keyed local data in terms of textual length and extract a similarly distributed random sample of texts from my local Chronicling America data. Then I hand-keyed this random stew, making sure it didn't coincidentally extract references to racial violence. As it turns out, it did one time. 499/500 random chunks of text didn't contain reference to local racial violence, but here's the article about a local lynching that I randomly found in the hand-keying process: https://tile.loc.gov/image-services/iiif/service:ndnp:scu:batch_scu_bogoff_ver01:data:sn84026907:00295862968:1912032001:0216/1337,286,1136,6072/1024,/0/default.jpg. That was quite surprising. And an example of why it's always worth it to hand-key stuff. I had thought the chances of randomly landing on a local report of racial violence were so miniscule that I almost skipped hand-keying the random stew, but the chances turned out to be 1/500 in this exercise.

Anyway, I began as always by loading the necessary libraries:

In [ ]:
import pandas as pd
from pathlib import Path
import os
import csv
import numpy as np
import html
import ipywidgets as w
from IPython.display import display

## 1) Get the range of verified_articles lengths

Then I set about analyzing my positive samples. To properly fine-tune BERT, I wanted a similarly distributed sample of negative examples, meaning that I had to review my positive samples first. I knew I had 300 local articles but some were as short as two or three sentences while others were full-page spreads. I didn't want to risk BERT parsing positive and negative examples based on text length or anything like that, so I worked with Codex to build the following quartile assessment of my positive examples in terms of their textual length.

It works like this: I tokenized the text, identified shortest and longest texts and overall median, then built quartiles of the token counts. Then I put each text into a bucket based on its quartile, giving me counts of texts grouped together by their relative lengths.

In [ ]:
df = pd.read_csv('results_df_progress.csv')

# Keep only non-null strings
s = df["verified_articles"].dropna().astype(str)

# Count tokens per row
token_counts = s.str.split().str.len()

# Basic stats
shortest = int(token_counts.min())
longest = int(token_counts.max())
overall_median = float(token_counts.median())

print("Shortest token length:", shortest)
print("Longest token length:", longest)
print("Overall median token length:", overall_median)

# Quartile cut points (0%, 25%, 50%, 75%, 100%)
quartiles = token_counts.quantile([0, 0.25, 0.50, 0.75, 1.0])

print("\nQuartile boundaries:")
print(quartiles)

# Put each row into a 25% bucket
buckets = pd.qcut(token_counts, q=4, duplicates="drop")

# Summary per bucket
bucket_summary = (
    pd.DataFrame({"token_count": token_counts, "bucket": buckets})
    .groupby("bucket", observed=True)["token_count"]
    .agg(["count", "min", "median", "max"])
)

print("\nBucket summary:")
print(bucket_summary)

The results are provided below. From this little step, I learned that my positive training data had quite the range of text lengths! The shortest was only 16 tokens. The longest, 4230. The median token length was 301. Using these distributions, I went onto build a similarly distributed random stew.

In [ ]:
'''
OUTPUT:
Shortest token length: 16
Longest token length: 4230
Overall median token length: 301.0

Quartile boundaries:
0.00      16.0
0.25      96.0
0.50     301.0
0.75     787.5
1.00    4230.0
Name: verified_articles, dtype: float64

Bucket summary:
                 count  min  median   max
bucket
(15.999, 96.0]      76   16    51.0    96
(96.0, 301.0]       74   97   185.5   300
(301.0, 787.5]      75  302   482.0   786
(787.5, 4230.0]     75  792  1407.0  4230
'''

## 2) extracting a random sample of texts from sundown_town_data

At this point, I needed to take these bucket ranges and extract texts from my Chronicling America data that fit these ranges. I didn't actually need to segment the Chronicling America text in any other particular way. Text lengths were all I needed to consider. This is because, in later steps, my fine-tuned BERT model would have to classify random segments of newspaper text, not clearly delineated newspaper articles. By creating a random sample that mirrored that quality, I would be fine-tuning the model on similar materials to what it would be classifying.

To do this, I worked with Codex to build the following process. In a nutshell, it first reads over the subdirectories in my sundown_town_data subset of Chronicling America. From these subdirectories, it picks random candidates–500 random pathways from the victim layer to the LCCN layer. In other words, it randomly selects the victim case and one associated newspaper. This ensures all 500 extracted texts come from different newspapers.

Then it randomly selects one OCR page from each newspaper candidate. It then takes a random chunk of text from that page. It takes chunks of random length, but ensures that the told number of random lengths fit the four quartile buckets we established above. In other words, it fills each "text-length bucket" with random texts of appropriate length.

I opted for 500 sample texts. That was a relatively arbitrary number, just going off vibes. I just felt that 300 positive examples and 500 negative examples would be enough to fine-tune BERT.

In [ ]:
def list_victim_lccn_branches(root_dir):
    root = Path(root_dir)
    branches = []

    for victim_entry in os.scandir(root):
        if not victim_entry.is_dir():
            continue

        for lccn_entry in os.scandir(victim_entry.path):
            if lccn_entry.is_dir():
                branches.append(Path(lccn_entry.path))

    return branches


def choose_random_txt_in_branch(branch_path, rng):
    """
    Pick one .txt uniformly at random from anywhere under a victim/lccn branch
    using reservoir sampling, while excluding macOS sidecar files like '._ocr.txt'.
    """
    chosen = None
    seen = 0

    for dirpath, _, filenames in os.walk(branch_path):
        for filename in filenames:
            if not filename.lower().endswith(".txt"):
                continue
            if filename.startswith("._"):
                continue
            if filename == ".DS_Store":
                continue

            seen += 1
            candidate = Path(dirpath) / filename
            if rng.integers(seen) == 0:
                chosen = candidate

    return chosen


def build_target_lengths(df, n_sample, rng):
    verified = df["verified_articles"].dropna().astype(str)
    token_counts = verified.str.split().str.len()

    if token_counts.empty:
        raise ValueError("df['verified_articles'] has no non-null string values.")

    labels = ["0-25%", "25-50%", "50-75%", "75-100%"]
    quartiles = token_counts.quantile([0, 0.25, 0.50, 0.75, 1.0]).tolist()

    # Force strictly increasing bin edges if quantiles tie
    edges = [int(np.floor(quartiles[0]))]
    for q in quartiles[1:]:
        q = int(np.ceil(q))
        if q <= edges[-1]:
            q = edges[-1] + 1
        edges.append(q)

    edges[-1] = max(edges[-1], int(token_counts.max()))

    buckets = pd.cut(
        token_counts,
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True,
    )

    proportions = buckets.value_counts(normalize=True).reindex(labels, fill_value=0)
    raw_counts = proportions * n_sample
    bucket_counts = np.floor(raw_counts).astype(int)

    remainder = n_sample - int(bucket_counts.sum())
    if remainder > 0:
        frac = (raw_counts - bucket_counts).sort_values(ascending=False)
        for label in frac.index[:remainder]:
            bucket_counts[label] += 1

    lengths_by_bucket = {
        label: token_counts[buckets == label].tolist()
        for label in labels
    }

    target_lengths = []
    target_buckets = []

    for label in labels:
        count = int(bucket_counts[label])
        lengths = lengths_by_bucket[label]
        if count == 0 or not lengths:
            continue

        sampled = rng.choice(lengths, size=count, replace=True)
        target_lengths.extend(sampled.tolist())
        target_buckets.extend([label] * count)

    targets = pd.DataFrame({
        "target_bucket": target_buckets,
        "target_token_count": target_lengths,
    })

    if len(targets) != n_sample:
        raise ValueError(f"Expected {n_sample} targets, got {len(targets)}.")

    return targets.sample(frac=1, random_state=int(rng.integers(1_000_000_000))).reset_index(drop=True)


def extract_random_segment(text, target_len, rng):
    tokens = text.split()
    if not tokens:
        return "", 0, 0, 0

    file_len = len(tokens)

    if file_len <= target_len:
        return " ".join(tokens), 0, file_len, file_len

    max_start = file_len - target_len
    start_idx = int(rng.integers(0, max_start + 1))
    segment = tokens[start_idx:start_idx + target_len]
    return " ".join(segment), start_idx, len(segment), file_len


def sample_random_branch_texts(
    df,
    root_dir="/Volumes/t5_evo_8tb/ChronAm/sundown_town_data",
    n_sample=500,
    random_state=42,
    out_csv="random_text_sample_500.csv",
):
    rng = np.random.default_rng(random_state)
    root = Path(root_dir)

    # 1) Build target lengths from verified_articles
    targets = build_target_lengths(df, n_sample=n_sample, rng=rng)

    # 2) List victim/lccn branches only
    branches = list_victim_lccn_branches(root)
    if not branches:
        raise ValueError("No victim/lccn branches found.")

    # Shuffle branch order once, then keep consuming until 500 valid rows
    branch_order = rng.permutation(len(branches))
    branch_cursor = 0

    rows = []
    used_txt_paths = set()

    skipped_no_txt = 0
    skipped_duplicate_txt = 0
    skipped_unreadable = 0
    skipped_bad_layout = 0
    skipped_empty_text = 0
    skipped_dot_underscore = 0

    while len(rows) < n_sample and branch_cursor < len(branch_order):
        branch_idx = branch_order[branch_cursor]
        branch_cursor += 1
        branch_path = branches[branch_idx]

        txt_path = choose_random_txt_in_branch(branch_path, rng)
        if txt_path is None:
            skipped_no_txt += 1
            continue

        # Extra safety guard
        if txt_path.name.startswith("._"):
            skipped_dot_underscore += 1
            continue

        if str(txt_path) in used_txt_paths:
            skipped_duplicate_txt += 1
            continue

        try:
            text = txt_path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            skipped_unreadable += 1
            continue

        rel_parts = txt_path.relative_to(root).parts
        if len(rel_parts) < 8:
            skipped_bad_layout += 1
            continue

        target_row = targets.iloc[len(rows)]
        target_len = int(target_row["target_token_count"])

        random_text, start_idx, actual_len, source_file_len = extract_random_segment(
            text=text,
            target_len=target_len,
            rng=rng,
        )

        if not random_text.strip():
            skipped_empty_text += 1
            continue

        victim, matched_lccn, year, month, date, ed, seq = rel_parts[:7]

        used_txt_paths.add(str(txt_path))
        rows.append({
            "victim": victim,
            "matched_lccn": matched_lccn,
            "year": year,
            "month": month,
            "date": date,
            "ed": ed,
            "seq": seq,
            "ocr_path": str(txt_path),
            "random_text": random_text,
            # optional diagnostics
            "target_bucket": target_row["target_bucket"],
            "target_token_count": target_len,
            "actual_segment_token_count": actual_len,
            "source_file_token_count": source_file_len,
            "segment_start_token": start_idx,
        })

    if len(rows) < n_sample:
        raise ValueError(
            f"Only produced {len(rows)} rows after checking all {len(branches)} victim/lccn branches. "
            f"Skipped: no_txt={skipped_no_txt}, duplicate_txt={skipped_duplicate_txt}, "
            f"unreadable={skipped_unreadable}, bad_layout={skipped_bad_layout}, "
            f"empty_text={skipped_empty_text}, dot_underscore={skipped_dot_underscore}."
        )

    result = pd.DataFrame(rows)

    # Final safety check
    if result["ocr_path"].str.split("/").str[-1].str.startswith("._").any():
        raise ValueError("Found a sampled file whose basename starts with '._'. Filtering failed.")

    output = result[
        ["victim", "matched_lccn", "year", "month", "date", "ed", "seq", "ocr_path", "random_text"]
    ].copy()

    # Quote all fields so long text is written safely
    output.to_csv(out_csv, index=False, quoting=csv.QUOTE_ALL)

    diagnostics = (
        result.groupby("target_bucket", observed=True)["actual_segment_token_count"]
        .agg(["count", "min", "median", "max"])
        .reset_index()
    )

    print(f"Saved {len(output)} rows to {out_csv}")
    print("\nSkip summary:")
    print({
        "no_txt": skipped_no_txt,
        "duplicate_txt": skipped_duplicate_txt,
        "unreadable": skipped_unreadable,
        "bad_layout": skipped_bad_layout,
        "empty_text": skipped_empty_text,
        "dot_underscore": skipped_dot_underscore,
    })

    print("\nValidation:")
    print("Rows:", len(output))
    print("Missing random_text values before save:", output["random_text"].isna().sum())
    print("Empty random_text strings before save:", (output["random_text"].str.strip() == "").sum())
    print("Sampled ._ files:", output["ocr_path"].str.split("/").str[-1].str.startswith("._").sum())

    return output, result, diagnostics


# Run it
output_df, full_result_df, diagnostics_df = sample_random_branch_texts(
    df,
    root_dir="/Volumes/t5_evo_8tb/ChronAm/sundown_town_data",
    n_sample=500,
    random_state=42,
    out_csv="random_text_sample_500.csv",
)

print("\nDiagnostics:")
print(diagnostics_df)

Below is the diagnostics report from the whole process. Across four buckets, I got 500 random chunks of newspaper text. Hooray! And just like my positive sample, they contained a similar length distributions. The most valuable metric here was the median lengths. In my positive sample, the buckets (quartiles) had median lengths of 51, 185, 482, and 1407. In my random sample, the median lengths are comparable, meaning that I got a similarly distributed sample overall.

In [ ]:
'''
OUTPUT:
Diagnostics:
  target_bucket  count  min  median   max
0         0-25%    127   16    54.0    96
1        25-50%    123   97   177.0   300
2        50-75%    125  323   488.0   786
3       75-100%    125  826  1407.0  3904
'''

And then I realized I didn't add the Chronicling America url link to this random sample. Oops! I once again built the urls in the same way I've done in previous notebooks.

In [ ]:
test_df = pd.read_csv('random_text_sample_500.csv')

# forgot to add the url column as per usual!
seq_no = test_df['seq'].astype(str).str.extract(r'(\d+)', expand=False)
month = test_df['month'].astype(str).str.zfill(2)
day = test_df['date'].astype(str).str.zfill(2)

test_df['chron_am_url'] = (
    'https://www.loc.gov/resource/'
    + test_df['matched_lccn'].astype(str)
    + '/'
    + test_df['year'].astype(str)
    + '-'
    + month
    + '-'
    + day
    + '/'
    + test_df['ed'].astype(str)
    + '/?sp='
    + seq_no.fillna('')
    + '&st=text&r=1,1,1,1'
)

test_df.to_csv('random_text_sample_500.csv', index=False)

## 3) Hand-key the random sample

At this point, I had my random sample, but I still wanted to do my due diligence and review it by hand. I didn't want any local reports of racial violence in it. That would throw off the model. So, as before, I reconfigured my hand-keying GUI and went through all 500 newspaper texts.

In [ ]:
# ---------------------------
# Config
# ---------------------------
DATA_PATH = "random_text_sample_500.csv"
TEXT_COL = "random_text"
STATUS_COL = "usability_status"   # values: pending, usable, ignore

REQUIRED_COLS = [
    "victim",
    "chron_am_url",
    TEXT_COL,
]

# ---------------------------
# Helpers
# ---------------------------
def atomic_save(_df, path):
    tmp = path + ".tmp"
    _df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def first_pending_pos(_df):
    idx = _df.index[_df[STATUS_COL].eq("pending")]
    return int(idx[0]) if len(idx) else 0

def safe_text(val):
    return "" if pd.isna(val) else str(val)

# ---------------------------
# Load / initialize dataframe
# ---------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Missing {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

for c in REQUIRED_COLS:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.reset_index(drop=True)

if STATUS_COL not in df.columns:
    df[STATUS_COL] = "pending"
else:
    valid = {"pending", "usable", "ignore"}
    df[STATUS_COL] = df[STATUS_COL].where(df[STATUS_COL].isin(valid), "pending")

atomic_save(df, DATA_PATH)

state = {"pos": first_pending_pos(df)}

# ---------------------------
# Widgets
# ---------------------------
progress = w.IntProgress(min=0, max=len(df), value=0, description="Done")
row_label = w.HTML()
victim_html = w.HTML()
url_html = w.HTML()
text_html = w.HTML()

decision = w.ToggleButtons(
    options=[("Usable", "usable"), ("Ignore", "ignore")],
    description="Status:",
    value="usable",
)

status = w.HTML()

btn_prev = w.Button(description="Previous")
btn_save = w.Button(description="Save")
btn_save_next = w.Button(description="Save + Next")
btn_next = w.Button(description="Next")
btn_first_pending = w.Button(description="First Pending")

# ---------------------------
# UI logic
# ---------------------------
def render():
    pos = state["pos"]
    row = df.iloc[pos]

    done = df[STATUS_COL].isin(["usable", "ignore"]).sum()
    progress.value = int(done)

    row_label.value = f"<b>Row {pos + 1} of {len(df)}</b>"

    victim_text = html.escape(safe_text(row["victim"]))
    url_text = safe_text(row["chron_am_url"]).strip()
    random_text = html.escape(safe_text(row[TEXT_COL]))

    victim_html.value = f"<b>Victim:</b> {victim_text}"

    if url_text:
        escaped_url = html.escape(url_text)
        url_html.value = f'<b>chron_am_url:</b> <a href="{escaped_url}" target="_blank">{escaped_url}</a>'
    else:
        url_html.value = "<b>chron_am_url:</b> "

    text_html.value = (
        f"<b>{TEXT_COL}</b>"
        "<div style='white-space:pre-wrap;border:1px solid #ddd;padding:8px;margin-top:4px;max-height:360px;overflow-y:auto;'>"
        f"{random_text}</div>"
    )

    row_status = row[STATUS_COL] if row[STATUS_COL] in ("usable", "ignore") else "pending"
    decision.value = "ignore" if row_status == "ignore" else "usable"

    status.value = ""

def save_current():
    pos = state["pos"]
    df.at[pos, STATUS_COL] = decision.value
    atomic_save(df, DATA_PATH)

    if decision.value == "usable":
        status.value = "<span style='color:green;'>Saved as Usable.</span>"
    else:
        status.value = "<span style='color:green;'>Saved as Ignore.</span>"

    return True

def on_prev(_):
    state["pos"] = max(0, state["pos"] - 1)
    render()

def on_next(_):
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()

def on_save(_):
    save_current()
    render()

def on_save_next(_):
    save_current()
    state["pos"] = min(len(df) - 1, state["pos"] + 1)
    render()

def on_first_pending(_):
    state["pos"] = first_pending_pos(df)
    render()

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_save.on_click(on_save)
btn_save_next.on_click(on_save_next)
btn_first_pending.on_click(on_first_pending)

ui = w.VBox([
    progress,
    row_label,
    victim_html,
    url_html,
    decision,
    text_html,
    w.HBox([btn_prev, btn_save, btn_save_next, btn_next, btn_first_pending]),
    status,
])

render()
display(ui)

That took a while! But it was faster than previous tagging efforts. I didn't need to designate the full texts of articles or anything. I just needed to scan the texts to make sure they didn't reference local events of racial violence.

After tagging, I ended up with 499/500 randomly drawn samples. The one that is not usable is row 416: Flute Clarke. The random extraction doesn't reference the Flute Clarke case, but a different lynching of three Black men as they were taken from one jail in the county to another. Here's the text of that one local report as it appears in the data:

Lynch Three Negroes For Using The Torch. Olar, March 13.?Three negroes in charge of two cons able3 on their way from Olar to Bamberg to be lodged in j tfa county jail were taken from the I officers by a mob at Odom's bridge, seven miles from this place, and shot to pieces this afternoon. The mob of 75 to 100 men surprised the two constables and quickly securing the three negroes finished their work in short order. The negroes were: Alfred D ;blin, 25 years of age; Richard Dublin, 30 years of age, and Peter Rivers, 10 years of age. All three of the negroes had confessed to attempting to burn the house of J. E. Cook, mayor of Olar, early yesterday morning.

And here's a link to the Chron Am newspaper clipping: https://www.loc.gov/resource/sn84026907/1912-03-20/ed-1/?sp=4&st=text&r=1,1,1,1